# 성 및 연령별 추계인구 분석 프로젝트

이 notebook은 성별 및 연령별 장래인구추계 데이터를 읽고, 전체 인구 변화와 연령별 인구 구조를 분석하는 빅데이터 프로젝트 예제입니다.

# 1. 라이브러리 불러오기

데이터 처리에는 pandas, 시각화에는 matplotlib, 파일 경로 처리에는 pathlib을 사용합니다. 한글 그래프가 깨지지 않도록 Windows 기본 한글 폰트인 Malgun Gothic을 설정합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 2. CSV 파일 읽기

장래인구추계 CSV 파일은 한글이 포함되어 있으므로 cp949 encoding을 우선 사용합니다. GitHub 저장소에서는 data/ 폴더의 파일을 사용하고, 로컬 PC에서는 기존 C:\data 경로도 fallback으로 지원합니다.

In [ ]:
# GitHub 저장소 안의 data 폴더를 기본 경로로 사용합니다.
file_path = Path("data/성_및_연령별_추계인구.csv")

# 로컬 PC에서 기존 C:\data 경로를 사용할 경우를 대비한 fallback입니다.
if not file_path.exists():
    file_path = Path(r"C:\data\성_및_연령별_추계인구.csv")

encodings = ["cp949", "euc-kr", "utf-8-sig", "utf-8"]
last_error = None

for enc in encodings:
    try:
        df = pd.read_csv(file_path, encoding=enc)
        print(f"성공적으로 읽은 encoding: {enc}")
        break
    except UnicodeDecodeError as e:
        last_error = e
else:
    raise last_error

df.head()

# 3. 데이터 구조 확인

데이터의 행과 열 개수, 열 이름, 자료형을 확인하여 분석 전에 전체 구조를 파악합니다.

In [ ]:
print("행과 열의 개수:", df.shape)
print("열 이름:")
print(df.columns.tolist())

df.info()
df.head()

# 4. 열 이름 정리

한국어 열 이름을 분석하기 쉬운 영어 변수명으로 변경합니다. 또한 2022년부터 2072년까지의 연도 열을 자동으로 찾아 숫자형으로 변환합니다.

In [ ]:
df = df.rename(columns={
    "가정별": "scenario",
    "성별": "sex",
    "연령별": "age_group"
})

# 2022-2072 형태의 연도 열을 자동으로 찾습니다.
year_cols = [col for col in df.columns if str(col).isdigit()]

# 연도 열을 숫자형으로 변환합니다.
df[year_cols] = df[year_cols].apply(pd.to_numeric, errors="coerce")

print("연도 열:", year_cols[:5], "...", year_cols[-5:])
df.head()

# 5. 주요 분류값 확인

추계 시나리오, 성별, 연령 구간에 어떤 값들이 들어 있는지 확인합니다.

In [ ]:
print("Scenario:")
display(df["scenario"].drop_duplicates())

print("Sex:")
display(df["sex"].drop_duplicates())

print("Age groups:")
display(df["age_group"].drop_duplicates())

# 6. 전체 인구 데이터 추출

성별=전체, 연령별=계 조건을 사용하여 전체 인구 추계 행만 추출하고, 연도별 인구 데이터프레임을 만듭니다.

In [ ]:
total_row = df[(df["sex"] == "전체") & (df["age_group"] == "계")].iloc[0]

total_population = pd.DataFrame({
    "year": [int(y) for y in year_cols],
    "population": total_row[year_cols].astype(int).values
})

total_population.head()

# 7. 전체 인구 변화 시각화

2022년부터 2072년까지 전체 인구가 어떻게 변화하는지 선 그래프로 확인합니다.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(total_population["year"], total_population["population"], marker="o", linewidth=2)
plt.title("전체 인구 추계")
plt.xlabel("연도")
plt.ylabel("인구")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8. long format 변환

wide format 데이터를 long format으로 변환합니다. long format은 연도별 필터링, 그룹 분석, 그래프 작성에 더 적합합니다.

In [ ]:
long_df = df.melt(
    id_vars=["scenario", "sex", "age_group"],
    value_vars=year_cols,
    var_name="year",
    value_name="population"
)

long_df["year"] = long_df["year"].astype(int)
long_df["population"] = pd.to_numeric(long_df["population"], errors="coerce")

long_df.head()

# 9. 성별 인구 비교

전체 연령 합계 기준으로 성별 인구 변화를 비교합니다.

In [ ]:
gender_total = long_df[long_df["age_group"] == "계"].copy()

plt.figure(figsize=(12, 5))

for sex_name, part in gender_total.groupby("sex"):
    plt.plot(part["year"], part["population"], marker="o", linewidth=2, label=sex_name)

plt.title("성별 인구 추계")
plt.xlabel("연도")
plt.ylabel("인구")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# 10. 특정 연도의 연령별 인구 선택

분석하고 싶은 연도를 지정한 뒤, 해당 연도의 연령별 인구 데이터를 추출합니다.

In [ ]:
selected_year = 2026

age_population = long_df[
    (long_df["sex"] == "전체") &
    (long_df["age_group"] != "계") &
    (long_df["year"] == selected_year)
].copy()

age_population.head()

# 11. 연령별 인구 막대그래프

선택한 연도의 연령 구간별 인구를 가로 막대그래프로 시각화합니다.

In [ ]:
plt.figure(figsize=(10, 8))
plt.barh(age_population["age_group"], age_population["population"])
plt.title(f"{selected_year}년 연령별 인구")
plt.xlabel("인구")
plt.ylabel("연령 구간")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# 12. 특정 연령 구간의 변화 분석

예시로 65-69세 연령 구간을 선택하여 연도별 인구 변화를 확인합니다.

In [ ]:
target_age_group = "65 - 69세"

target_age = long_df[
    (long_df["sex"] == "전체") &
    (long_df["age_group"] == target_age_group)
].copy()

plt.figure(figsize=(12, 5))
plt.plot(target_age["year"], target_age["population"], marker="o", linewidth=2)
plt.title(f"연령 구간별 인구 추계: {target_age_group}")
plt.xlabel("연도")
plt.ylabel("인구")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 13. 재사용 가능한 필터 함수

성별, 연령 구간, 시나리오를 입력하면 조건에 맞는 인구 데이터를 반환하는 함수를 만듭니다.

In [ ]:
def get_population(sex="전체", age_group="계", scenario=None):
    result = long_df[
        (long_df["sex"] == sex) &
        (long_df["age_group"] == age_group)
    ].copy()
    
    if scenario is not None:
        result = result[result["scenario"] == scenario]
    
    return result.sort_values("year")

# 예시: 전체 인구
get_population(sex="전체", age_group="계").head()

# 14. 분석 결과 저장

long format으로 변환한 데이터를 outputs/population_long_format.csv 파일로 저장합니다.

In [ ]:
Path("outputs").mkdir(exist_ok=True)
output_path = Path("outputs/population_long_format.csv")
long_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장한 파일:", output_path.resolve())